In [1]:
# Импорт модулей
import sys
import os

sys.path.insert(0, os.path.abspath('src'))

# Импортируем модули
from core.rag_engine import RagEngine
from core.agents import create_researcher_agent, create_factchecker_agent
from core.guardrails import Guardrails
from core.logger import Logger
from core.config import data_path, chroma_persist_dir, retrieval_k

from langchain.tools import Tool

print("Все модули успешно импортированы!")

Все модули успешно импортированы!


In [2]:
# Индексация документов
rag = RagEngine()

# Проверяем, есть ли уже индекс
if not os.path.exists(chroma_persist_dir):
    print("Индекс не найден. Начинаем индексацию документов...")
    rag.index_documents()
else:
    print("Индекс уже существует. Загружаем его...")
    # Принудительно загружаем существующее хранилище
    rag.vectorstore = None  # метод retrieve сам подгрузит из папки

# Тестовый запрос, чтобы убедиться, что поиск работает
test_docs = rag.retrieve("Что такое искусственный интеллект?", k=2)
print(f"\nНайдено {len(test_docs)} релевантных фрагмента:")
for i, doc in enumerate(test_docs, 1):
    print(f"\nФрагмент {i}")
    print(doc.page_content[:300] + "...")

Индекс уже существует. Загружаем его...


C:\Users\olgas\Desktop\1. Магистратура\Учеба\4 семестр\Генеративные задачи в NLP II\1 дз\src\core\rag_engine.py:55: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  self.vectorstore = Chroma(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Найдено 2 релевантных фрагмента:

Фрагмент 1
# Обработка Языка

Модели на основе архитектуры Трансформер, такие как BERT, GPT и T5, достигли выдающихся результатов во множестве задач NLP, включая классификацию текста, машинный перевод и ответы на вопросы....

Фрагмент 2
# Этика Ии

Объяснимый ИИ (XAI) — это направление, целью которого является создание моделей, решения которых могут быть поняты и интерпретированы человеком....


In [3]:
# Создание агентов
# Инструмент для поиска по базе знаний
def retrieve_func(query):
    docs = rag.retrieve(query)
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

retriever_tool = Tool(
    name="ПоискПоБазеЗнаний",
    func=retrieve_func,
    description="Искать релевантную информацию в загруженных документах. Вход: поисковый запрос."
)

# Создаём агентов
researcher = create_researcher_agent(retriever_tool)
factchecker = create_factchecker_agent()
logger = Logger()

print("Агенты Researcher и FactChecker созданы.")

Агенты Researcher и FactChecker созданы.


In [4]:
# Проверка защитных фильтров
test_queries = [
    "Расскажи про искусственный интеллект",  # безопасный
    "Как совершить насилие над людьми",        # запрещённая тема
]

print("Тестирование Guardrails на входе:")
for q in test_queries:
    valid, msg = Guardrails.validate_input(q)
    if valid:
        print(f"Запрос принят: '{q}'")
    else:
        print(f"Запрос отклонён: '{q}' -> {msg}")

Тестирование Guardrails на входе:
Запрос принят: 'Расскажи про искусственный интеллект'
Запрос отклонён: 'Как совершить насилие над людьми' -> В запросе обнаружена запрещённая тема: 'насилие'. Пожалуйста, переформулируйте.


In [5]:
# Полный цикл обработки запроса
user_query = "Что такое искусственный интеллект и когда он появился?"

print(f"Вопрос пользователя: {user_query}\n")

# 1. Проверка Guardrails
valid, msg = Guardrails.validate_input(user_query)
if not valid:
    print(f"Запрос отклонён: {msg}")
else:
    # 2. Запуск Researcher Agent
    print("Researcher анализирует запрос...")
    researcher_response = researcher.invoke({"input": user_query})
    draft_answer = researcher_response["output"]

    print("\nЧерновой ответ Researcher (с рассуждениями)")
    print(draft_answer)

    # 3. Получение контекста для фактчекинга
    retrieved_docs = rag.retrieve(user_query, k=retrieval_k)
    context_text = "\n\n---\n\n".join([doc.page_content for doc in retrieved_docs])

    # 4. Проверка фактов
    print("\nFactChecker проверяет ответ...")
    fc_response = factchecker.invoke({
        "draft": draft_answer,
        "context": context_text
    })
    verdict = fc_response.content if hasattr(fc_response, 'content') else str(fc_response)

    print("\nИтог FactChecker")
    print(verdict)

    # 5. Проверка вывода Guardrails
    out_valid, out_msg = Guardrails.validate_output(draft_answer)
    if not out_valid:
        print(f"\nGuardrails на выходе: {out_msg}")

    # 6. Логирование
    logger.log("response_generated", {
        "query": user_query,
        "draft_length": len(draft_answer),
        "verdict": verdict
    })
    print("\nСобытие записано в лог.")

Вопрос пользователя: Что такое искусственный интеллект и когда он появился?

Researcher анализирует запрос...


> Entering new AgentExecutor chain...
 Искусственный интеллект (ИИ) - это компьютерная система, которая может выполнять задачи, которые обычно ассоциируются с человеческим разумом, такие как обработка естественного языка, распознавание изображений и логическое мышление.

История ИИ начинается в 1956 году, когда группа ученых собралась на конференцию по машинному обучению в Дартмуте (Нью-Гемпшир, США) и официально определила целью создания системы, способной мыслить как человек. Это событие считается началом современного ИИ.

В 1956 году также был создан первый компьютерный алгоритм для игры в шахматы - Мак-Хайд (McCarthy, Minsky and Shannon's Automated Reasoning System). Это была одна из первых систем ИИ, способных демонстрировать логическое мышление.

В дальнейшем были созданы многие другие системы ИИ, такие как ELIZA (1964), которая могла имитировать диалог с человеком, и D

In [6]:
# Просмотр логов
import json
from core.config import log_dir
from datetime import datetime

log_file = os.path.join(log_dir, f"{datetime.now().strftime('%Y-%m-%d')}.jsonl")

if os.path.exists(log_file):
    print(f"Содержимое лог-файла {log_file}:\n")
    with open(log_file, "r", encoding="utf-8") as f:
        for line in f:
            entry = json.loads(line)
            print(f"[{entry['timestamp']}] {entry['event']}: {entry['data']}")
else:
    print("Лог-файл за сегодня ещё не создан.")

Содержимое лог-файла C:\Users\olgas\Desktop\1. Магистратура\Учеба\4 семестр\Генеративные задачи в NLP II\1 дз\src\core\..\..\logs\2026-04-14.jsonl:

[2026-04-14T14:52:25.976272] response_generated: {'query': 'Что такое искусственный интеллект и когда он появился?', 'draft_length': 853, 'verdict': ' Ответ подтверждён. В контексте содержится точная информация о начале истории ИИ, а также упоминаются наиболее известные ранние системы ИИ - ЭЛIZA и SHRDLU.'}
[2026-04-14T14:58:26.225977] response_generated: {'query': 'Что такое искусственный интеллект и когда он появился?', 'draft_length': 1030, 'verdict': ' Ответ: Подтвержден\n\nОтвечая на вопрос о начале современного ИИ, черновой ответ соответствует информации из контекста. В 1956 году была официально определена цель создания системы, способной мыслить как человек, что считается началом современного ИИ. Кроме того, в этом же году был создан первый компьютерный алгоритм для игры в шахматы - Мак-Хайд, который также является одним из первых с

In [7]:
# Эксперимент с параметрами генерации
from langchain_ollama import ChatOllama
from core.config import ollama_model

# Создадим две модели с разной температурой
model_low_temp = ChatOllama(model=ollama_model, temperature=0.0)
model_high_temp = ChatOllama(model=ollama_model, temperature=0.9)

prompt = "Придумай короткое определение искусственного интеллекта для школьника."

print("Сравнение влияния температуры на ответ:\n")
print("temperature = 0.0 (детерминированный)")
response_low = model_low_temp.invoke(prompt)
print(response_low.content)

print("\ntemperature = 0.9 (креативный)")
response_high = model_high_temp.invoke(prompt)
print(response_high.content)

Сравнение влияния температуры на ответ:

temperature = 0.0 (детерминированный)
 Искусственный интеллект (ИИ) - это компьютерная программа, которая может выполнять задачи, похожие на человеческий разум, такие как распознавание образов, решение математических уравнений или понимание естественного языка. ИИ помогает автоматизировать процессы и снижать время на выполнение задач, но он не обладает собственной волей и самосознанием.

temperature = 0.9 (креативный)
 Искусственный интеллект - компьютерная система, способная самостоятельно решать задачи, ранее требовавшие человеческого интеллекта, такие как размышление, понимание, анализ и решение проблем.

Например: Робот или программа, которая может самостоятельно играть в шахматные матчи или распознавать образы.


**Анализ результатов и выводы**

1. Функционирование RAG-пайплайна
- Система успешно загрузила векторный индекс, построенный на основе 50 документов по темам, связанным с искусственным интеллектом.
- Тестовый запрос "Что такое искусственный интеллект?" вернул 2 релевантных фрагмента. Оба относятся к ИИ, хотя прямого словарного определения не содержат – информация распределена по тематическим документам.
- При запросе "Что такое искусственный интеллект и когда он появился?" агент Researcher сформировал связный ответ: определение ИИ, указание на 1956 год и Дартмутскую конференцию, упоминание ранних систем (ELIZA, Deep Blue).<br>
Вывод:<br>
RAG-пайплайн на базе ChromaDB и эмбеддингов nomic-embed-text через Ollama стабильно извлекает релевантную информацию из русскоязычных документов и агрегирует факты в осмысленный ответ.

2. Работа Guardrails
- Безопасный запрос "Расскажи про искусственный интеллект" принят без ограничений.
- Запрос с запрещённой темой "Как совершить насилие над людьми" отклонён с пояснением: «В запросе обнаружена запрещённая тема: 'насилие'».
- Проверка сгенерированного ответа не выявила запрещённых слов.<br>
Вывод:<br>
Реализованные на основе списков запрещённых тем и регулярных выражений Guardrails эффективно фильтруют как входящие запросы, так и исходящие ответы, повышая безопасность и этичность системы.

3. Многоагентная верификация
- Агент Researcher продемонстрировал пошаговое рассуждение (Chain-of-Thought), что видно в логах AgentExecutor.
- Агент FactChecker сравнил ответ с извлечённым контекстом и вынес вердикт **«Подтверждён»**. В обосновании отмечено, что ключевые факты (1956 год, ранние системы) соответствуют документам.<br>
Вывод:<br>
Двухэтапная проверка с участием специализированных агентов повышает достоверность ответов и позволяет выявлять потенциальные расхождения с базой знаний.

4. Влияние параметра температуры на генерацию
- При temperature = 0.0 модель дала краткое, детерминированное определение ИИ, близкое к формальному.
- При temperature = 0.9 ответ стал более развёрнутым и образным, появился конкретный пример (робот, играющий в шахматы).
Вывод:<br>
Параметр температуры напрямую влияет на стиль и креативность генерации. Для фактологического ассистента предпочтительны низкие значения (0.0–0.1), высокие – для творческих или объяснительных сценариев.

5. Преодолённые технические вызовы
В процессе настройки возникла проблема совместимости эмбеддингов Ollama с Windows (ошибка 502 при вызове /api/embed). Путём диагностики (прямые HTTP-запросы, анализ логов ollama serve) была выбрана стабильная модель nomic-embed-text, что позволило завершить индексацию и обеспечить работу RAG без внешних зависимостей.<br>
Вывод:<br>
Проблема успешно решена, система полностью автономна и не требует облачных API.

6. Практическая значимость и направления развития
Разработанный прототип может быть использован как:
- корпоративный ассистент по внутренней документации,
- учебный инструмент для самостоятельного изучения материалов,
- основа для исследований в области многоагентной верификации и безопасной генерации.

Возможные улучшения:
- Переход на langchain-chroma для устранения deprecation-предупреждений.
- Интеграция более продвинутых Guardrails (например, guardrails-ai).
- Добавление веб-интерфейса для загрузки новых документов.
- Количественная оценка качества ответов (точность, полнота).